In [ ]:
# ============================================================
# FLUX.1 [schnell] - Open Pretrained Diffusion Model Inference
# ============================================================
# Model: FLUX.1 [schnell] by Black Forest Labs
# Repo: https://github.com/black-forest-labs/flux
# Weights: https://huggingface.co/black-forest-labs/FLUX.1-schnell
# License: Apache-2.0
# GPU requirement: ~24GB VRAM (hoặc ~12GB với float16/quantized)

In [ ]:
# !pip install diffusers transformers torch accelerate sentencepiece protobuf

In [ ]:
import torch
import matplotlib.pyplot as plt
from diffusers import FluxPipeline

In [ ]:
# Load FLUX.1-schnell model
# schnell = fast in German, optimized for speed (4 steps inference)

In [ ]:
pipe = FluxPipeline.from_pretrained(
    "black-forest-labs/FLUX.1-schnell",
    torch_dtype=torch.bfloat16
)
pipe = pipe.to("cuda")

In [ ]:
# Nếu GPU < 24GB, bật CPU offload:
# pipe.enable_model_cpu_offload()

In [ ]:
# ============================================================
# Text-to-Image: Generate ảnh đơn giản từ 1 prompt
# ============================================================

In [ ]:
prompt = "A cute golden retriever puppy playing in a field of sunflowers, sunny day, photorealistic"

In [ ]:
image = pipe(
    prompt=prompt,
    num_inference_steps=4,
    guidance_scale=0.0,
    height=512,
    width=512,
    generator=torch.Generator("cuda").manual_seed(42)
).images[0]

In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.title("FLUX.1 [schnell] - Text-to-Image")
plt.axis("off")
plt.show()

In [ ]:
# ============================================================
# Generate nhiều ảnh với các prompt khác nhau
# ============================================================

In [ ]:
prompts = [
    "A majestic mountain landscape at sunset with a crystal clear lake reflection, 4k photography",
    "Portrait of an elderly Vietnamese woman wearing traditional ao dai, warm smile, studio lighting",
    "Abstract art with flowing neon colors and geometric shapes, digital art style",
    "A futuristic cyberpunk city street at night with neon signs and rain reflections",
]

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 6))

for i, prompt in enumerate(prompts):
    image = pipe(
        prompt=prompt,
        num_inference_steps=4,
        guidance_scale=0.0,
        height=512,
        width=512,
        generator=torch.Generator("cuda").manual_seed(42)
    ).images[0]
    axes[i].imshow(image)
    axes[i].set_title(f"Prompt {i+1}", fontsize=12)
    axes[i].axis("off")

plt.suptitle("FLUX.1 [schnell] - Various prompts", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
for i, p in enumerate(prompts):
    print(f"Prompt {i+1}: {p}")

In [ ]:
# ============================================================
# So sánh kết quả với các num_inference_steps khác nhau
# FLUX.1-schnell được tối ưu cho 1-4 steps
# ============================================================

In [ ]:
prompt = "A beautiful Japanese garden with cherry blossoms and a wooden bridge over a koi pond"
steps_list = [1, 2, 3, 4]

In [ ]:
fig, axes = plt.subplots(1, len(steps_list), figsize=(24, 6))

for i, steps in enumerate(steps_list):
    image = pipe(
        prompt=prompt,
        num_inference_steps=steps,
        guidance_scale=0.0,
        height=512,
        width=512,
        generator=torch.Generator("cuda").manual_seed(42)
    ).images[0]
    axes[i].imshow(image)
    axes[i].set_title(f"Steps = {steps}", fontsize=14)
    axes[i].axis("off")

plt.suptitle(f"Effect of num_inference_steps\nPrompt: \"{prompt}\"", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# So sánh kết quả với các seed khác nhau (diversity)
# Cùng 1 prompt nhưng seed khác → ảnh khác nhau
# ============================================================

In [ ]:
prompt = "A cat wearing a tiny top hat and monocle, sitting on a throne, oil painting style"
seeds = [0, 42, 123, 999]

In [ ]:
fig, axes = plt.subplots(1, len(seeds), figsize=(24, 6))

for i, seed in enumerate(seeds):
    image = pipe(
        prompt=prompt,
        num_inference_steps=4,
        guidance_scale=0.0,
        height=512,
        width=512,
        generator=torch.Generator("cuda").manual_seed(seed)
    ).images[0]
    axes[i].imshow(image)
    axes[i].set_title(f"Seed = {seed}", fontsize=14)
    axes[i].axis("off")

plt.suptitle(f"Same prompt, different seeds (diversity)\nPrompt: \"{prompt}\"", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# So sánh các resolution khác nhau
# ============================================================

In [ ]:
prompt = "An astronaut riding a horse on Mars, cinematic lighting, detailed"
resolutions = [(256, 256), (512, 512), (768, 768), (1024, 1024)]

In [ ]:
images = []
for h, w in resolutions:
    image = pipe(
        prompt=prompt,
        num_inference_steps=4,
        guidance_scale=0.0,
        height=h,
        width=w,
        generator=torch.Generator("cuda").manual_seed(42)
    ).images[0]
    images.append(image)

In [ ]:
fig, axes = plt.subplots(1, len(resolutions), figsize=(24, 6))
for i, (image, (h, w)) in enumerate(zip(images, resolutions)):
    axes[i].imshow(image)
    axes[i].set_title(f"{w}x{h}", fontsize=14)
    axes[i].axis("off")

plt.suptitle(f"Effect of resolution\nPrompt: \"{prompt}\"", fontsize=14)
plt.tight_layout()
plt.show()